<a href="https://colab.research.google.com/github/harry-8818/-From-Neural-Network-Foundations-to-Multi-Object-Tracking/blob/main/YOLO_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
yaml_content = """
tracker_type: botsort
track_high_thresh: 0.3
track_low_thresh: 0.05
new_track_thresh: 0.25
track_buffer: 90
with_reid: True
model: auto
appearance_thresh: 0.35
proximity_thresh: 0.5
gmc_method: sparseOptFlow
match_thresh: 0.8
fuse_score: True
"""
with open('most_optimal.yaml', 'w') as f:
        f.write(yaml_content.strip())
print("Done,yaml")

In [ ]:

import os
import cv2
import glob
import numpy as np
from ultralytics import YOLO

sequences = ["ref"]
base_file_path = '/content/Week4/Week 4/data/'
max_interpolation_gap = 30

def interpolate_tracks(input_path,output_path,max_gap):
  try :
    data = np.loadtxt(input_path,delimiter=',')
    if (len(data) == 0) :
      return
  except :
    return
  data = data[np.lexsort((data[:,0],data[:,1]))]
  interpolated_data = []
  unique_ids = np.unique(data[:,1])
  for id in unique_ids :
    id_data = data[data[:,1] == id]
    frames = id_data[:,0]
    for i in range(len(frames)-1) :
      interpolated_data.append(id_data[i])
      gap = int(frames[i+1] - frames[i])
      if ( 1 < gap <= max_gap) :
        box1 = id_data[i,2:6]
        box2 = id_data[i+1,2:6]
        for j in range(1,gap) :
          interpolated_frame = frames[i] + j
          weight = j/gap
          interpolated_box = box1*(1-weight) + box2*weight
          interpolated_data.append([interpolated_frame,id] + list(interpolated_box)+[-1,-1,-1,-1])
    interpolated_data.append(id_data[-1])
  interpolated_data = np.array(interpolated_data)
  interpolated_data = interpolated_data[interpolated_data[:,0].argsort()]
  np.savetxt(output_path,interpolated_data,fmt='%d,%d,%.2f,%.2f,%.2f,%.2f,%d,%d,%d,%d')

def process_sequence(model,sequence_name):
    sequence_folder_path = os.path.join(base_file_path,sequence_name,'img')
    raw_output_file_path = f"{sequence_name}_raw.txt"
    output_file_path = f"{sequence_name}.txt"
    img_paths = glob.glob(os.path.join(sequence_folder_path,'*.jpg'))
    img_paths.sort()
    if not img_paths:
        return
    print(f"Gnerating the text output file for {sequence_name} folder in Week 4 data ->\n")
    current_frame_number = 1
    with open(raw_output_file_path,'w') as f:
        for img_path in img_paths:
          current_frame = cv2.imread(img_path)
          results = model.track(current_frame,tracker="most_optimal.yaml",persist=True,classes=[0],imgsz=1920,conf=0.10,iou=0.75)
          if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xywh.cpu().numpy()
            ids = results[0].boxes.id.int().cpu().tolist()
            for box,id in zip(boxes,ids):
              x,y,w,h = box
              bb_left,bb_top = int(x - w / 2),int(y - h / 2)
              bb_width,bb_height = int(w),int(h)
              f.write(f"{current_frame_number},{id},{bb_left},{bb_top},{bb_width},{bb_height},-1,-1,-1,-1\n")
          current_frame_number += 1
    print(f"Sequence {sequence_name} is completed and saved to {output_file_path}")
    interpolate_tracks(raw_output_file_path,output_file_path,max_interpolation_gap)
    if os.path.exists(raw_output_file_path):
        os.remove(raw_output_file_path)

def main():
    model = YOLO("yolo11x.pt")
    for seq in sequences:
        process_sequence(model,seq)
if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import glob
import numpy as np
from ultralytics import YOLO

sequences = ["01","02","03"]
base_file_path = '/content/Week4/Week 4/data/'
max_interpolation_gap = 30

def interpolate_tracks(input_path,output_path,max_gap):
  try :
    data = np.loadtxt(input_path,delimiter=',')
    if (len(data) == 0) :
      return
  except :
    return
  data = data[np.lexsort((data[:,0],data[:,1]))]
  interpolated_data = []
  unique_ids = np.unique(data[:,1])
  for id in unique_ids :
    id_data = data[data[:,1] == id]
    frames = id_data[:,0]
    for i in range(len(frames)-1) :
      interpolated_data.append(id_data[i])
      gap = int(frames[i+1] - frames[i])
      if ( 1 < gap <= max_gap) :
        box1 = id_data[i,2:6]
        box2 = id_data[i+1,2:6]
        for j in range(1,gap) :
          interpolated_frame = frames[i] + j
          weight = j/gap
          interpolated_box = box1*(1-weight) + box2*weight
          interpolated_data.append([interpolated_frame,id] + list(interpolated_box)+[-1,-1,-1,-1])
    interpolated_data.append(id_data[-1])
  interpolated_data = np.array(interpolated_data)
  interpolated_data = interpolated_data[interpolated_data[:,0].argsort()]
  np.savetxt(output_path,interpolated_data,fmt='%d,%d,%.2f,%.2f,%.2f,%.2f,%d,%d,%d,%d')

def process_sequence(model,sequence_name):
    sequence_folder_path = os.path.join(base_file_path,sequence_name,'img')
    raw_output_file_path = f"{sequence_name}_raw.txt"
    output_file_path = f"{sequence_name}.txt"
    img_paths = glob.glob(os.path.join(sequence_folder_path,'*.jpg'))
    img_paths.sort()
    if not img_paths:
        return
    print(f"Gnerating the text output file for {sequence_name} - \n")
    current_frame_number = 1
    with open(raw_output_file_path,'w') as f:
        for img_path in img_paths:
          current_frame = cv2.imread(img_path)
          results = model.track(current_frame,tracker="paper_optimal.yaml",persist=True,classes=[0],imgsz=1920,conf=0.10,iou=0.75)
          if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xywh.cpu().numpy()
            ids = results[0].boxes.id.int().cpu().tolist()
            for box,id in zip(boxes,ids):
              x,y,w,h = box
              bb_left,bb_top = int(x - w / 2),int(y - h / 2)
              bb_width,bb_height = int(w),int(h)
              f.write(f"{current_frame_number},{id},{bb_left},{bb_top},{bb_width},{bb_height},-1,-1,-1,-1\n")
          current_frame_number += 1
    print(f"Sequence {sequence_name} is completed and saved to {output_file_path}")
    interpolate_tracks(raw_output_file_path,output_file_path,max_interpolation_gap)
    if os.path.exists(raw_output_file_path):
        os.remove(raw_output_file_path)

def main():
    for seq in sequences:
      model = YOLO("yolo11x.pt")
      process_sequence(model,seq)
if __name__ == "__main__":
    main()